# Master GL 통합 데이터셋 구축
SAP 7개 테이블(BSEG, BKPF, SKA1, SKAT, KNA1, LFA1, CSKT) 병합

In [19]:
# 0. 라이브러리 & 경로 설정
import pandas as pd
import numpy as np
import pathlib
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = pathlib.Path().resolve()  # 노트북과 동일 디렉토리
print(f'작업 경로: {BASE_DIR}')

# ID성 필드 -> str 로드 (지수표기 방지, 앞자리 0 보존)
STR_COLS = {
    'bseg': ['bukrs', 'belnr', 'gjahr', 'buzei', 'saknr', 'hkont',
             'kunnr', 'lifnr', 'kostl', 'kokrs', 'aufnr'],
    'bkpf': ['bukrs', 'belnr', 'gjahr', 'stblg', 'stjah'],
    'ska1': ['saknr', 'ktopl'],
    'skat': ['saknr', 'ktopl'],
    'kna1': ['kunnr'],
    'lfa1': ['lifnr'],
    'cskt': ['kostl', 'kokrs'],
}


작업 경로: G:\내 드라이브\03_DArt-B\12_정규스터디\project


In [20]:
# 1. 데이터 로드
def load_csv(name):
    path = BASE_DIR / f'{name}.csv'
    str_cols = STR_COLS.get(name, [])
    df = pd.read_csv(path, dtype={c: str for c in str_cols},
                     encoding='utf-8', low_memory=False)
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].str.strip().replace({'nan': np.nan, '': np.nan})
    return df

bseg = load_csv('bseg')
bkpf = load_csv('bkpf')
ska1 = load_csv('ska1')
skat = load_csv('skat')
kna1 = load_csv('kna1')
lfa1 = load_csv('lfa1')
cskt = load_csv('cskt')

print('[로드 완료]')
for n, d in [('BSEG',bseg),('BKPF',bkpf),('SKA1',ska1),
              ('SKAT',skat),('KNA1',kna1),('LFA1',lfa1),('CSKT',cskt)]:
    print(f'  {n:5s}: {len(d):>8,}행  x  {d.shape[1]}열')


[로드 완료]
  BSEG :  332,106행  x  356열
  BKPF :  150,057행  x  125열
  SKA1 :  146,692행  x  21열
  SKAT :  782,051행  x  10열
  KNA1 :   20,209행  x  192열
  LFA1 :    3,035행  x  145열
  CSKT :   26,977행  x  11열


In [21]:
# 2. BSEG 사전 정제 (시스템 노이즈 제거)
bseg['dmbtr'] = pd.to_numeric(bseg['dmbtr'], errors='coerce').fillna(0)
raw_cnt = len(bseg)

# SAP BSEG 계정코드 통합:
#   koart=S/M (G/L, 자재) -> 계정코드가 saknr 대신 hkont에 저장됨
#   koart=K/D (벤더/고객) -> 조정계정이 saknr에 저장됨
#   -> saknr이 없으면 hkont로 채워 전 행에서 계정코드를 확보
bseg['saknr'] = bseg['saknr'].fillna(bseg['hkont'])

# 금액 0 or 계정코드 없는 행 제거 (시스템 생성 합계행 및 공백행)
bseg = bseg[
    (bseg['dmbtr'] != 0) &
    (bseg['hkont'].notna())
].copy()

bseg_cnt = len(bseg)  # 이후 검증에 사용
print(f'[BSEG 필터링]  {raw_cnt:,}행  ->  {bseg_cnt:,}행  (노이즈 제거 {raw_cnt - bseg_cnt:,}행)')


[BSEG 필터링]  332,106행  ->  332,103행  (노이즈 제거 3행)


In [22]:
# 3. 마스터 테이블 최적화

# SKAT: 영문(spras='E') 계정명, 계정코드별 최신 1건
skat_filt = skat[skat['spras'] == 'E'].copy()
if len(skat_filt) == 0:
    print('[SKAT] spras=E 없음 -> 전체 언어 사용')
    skat_filt = skat.copy()
skat_filt = (
    skat_filt
    .sort_values('recordstamp', ascending=False)
    .drop_duplicates(subset='saknr', keep='first')
)
print(f'[SKAT] 영문 계정명 {len(skat_filt):,}건')

# CSKT: 원가센터명, datbi 기준 최신 1건 (1:1 매핑 보장)
cskt_filt = cskt[cskt['spras'] == 'E'].copy() if 'E' in cskt['spras'].values else cskt.copy()
if 'E' not in cskt['spras'].values:
    print('[CSKT] spras=E 없음 -> 전체 언어 사용')
cskt_filt['datbi'] = pd.to_datetime(cskt_filt['datbi'], errors='coerce')
cskt_filt = (
    cskt_filt
    .sort_values('datbi', ascending=False)
    .drop_duplicates(subset='kostl', keep='first')
)
print(f'[CSKT] 최신 원가센터명 {len(cskt_filt):,}건')

# KNA1 / LFA1: 거래처코드별 중복 제거
kna1_slim = (
    kna1[['kunnr', 'name1']].sort_values('name1')
    .drop_duplicates(subset='kunnr', keep='first')
    .rename(columns={'name1': '_kna1_name'})
)
lfa1_slim = (
    lfa1[['lifnr', 'name1']].sort_values('name1')
    .drop_duplicates(subset='lifnr', keep='first')
    .rename(columns={'name1': '_lfa1_name'})
)
print(f'[KNA1] 고객 {len(kna1_slim):,}건  /  [LFA1] 벤더 {len(lfa1_slim):,}건')


[SKAT] 영문 계정명 16,350건
[CSKT] 최신 원가센터명 1,473건
[KNA1] 고객 19,942건  /  [LFA1] 벤더 2,590건


In [23]:
# 4. 테이블 병합

# Step 1: BSEG(라인) + BKPF(헤더)  |  키: bukrs, belnr, gjahr
# BKPF는 병렬회계/변경이력으로 동일 전표번호가 중복될 수 있음 -> 최신 1건만 사용
bkpf_cols = ['bukrs', 'belnr', 'gjahr', 'blart', 'budat', 'bldat',
             'stblg', 'stjah', 'xreversal', 'waers', 'recordstamp']
bkpf_dedup = (
    bkpf[bkpf_cols]
    .sort_values('recordstamp', ascending=False)
    .drop_duplicates(subset=['bukrs', 'belnr', 'gjahr'], keep='first')
    .drop(columns='recordstamp')
)
print(f'BKPF 중복제거: {len(bkpf):,}행 -> {len(bkpf_dedup):,}행')
df = bseg.merge(bkpf_dedup, on=['bukrs', 'belnr', 'gjahr'], how='left')

# Step 2: + SKA1 (계정 BS/PL 구분)  |  키: saknr
# SKA1은 ktopl(계정과목표)별로 동일 saknr 중복 -> 최신 recordstamp 기준 1건
ska1_dedup = (
    ska1[['saknr', 'xbilk', 'ktoks', 'recordstamp']]
    .sort_values('recordstamp', ascending=False)
    .drop_duplicates(subset='saknr', keep='first')
    .drop(columns='recordstamp')
)
print(f'SKA1 중복제거: {len(ska1):,}행 -> {len(ska1_dedup):,}행')
df = df.merge(ska1_dedup, on='saknr', how='left')

# Step 3: + SKAT (계정명)  |  키: saknr
df = df.merge(
    skat_filt[['saknr', 'txt50']].rename(columns={'txt50': '_계정명'}),
    on='saknr', how='left'
)

# Step 4: + KNA1 (고객명)  |  키: kunnr
df = df.merge(kna1_slim, on='kunnr', how='left')

# Step 5: + LFA1 (벤더명)  |  키: lifnr
df = df.merge(lfa1_slim, on='lifnr', how='left')

# Step 6: 통합거래처명 단일 필드 (고객 우선, 없으면 벤더)
df['_통합거래처명'] = df['_kna1_name'].fillna(df['_lfa1_name'])
df.drop(columns=['_kna1_name', '_lfa1_name'], inplace=True)

# Step 7: + CSKT (원가센터명/부서명)  |  키: kostl
df = df.merge(
    cskt_filt[['kostl', 'ltext']].rename(columns={'ltext': '_원가센터명'}),
    on='kostl', how='left'
)

print(f'[병합 완료]  {len(df):,}행  x  {df.shape[1]}열')


BKPF 중복제거: 150,057행 -> 136,891행
SKA1 중복제거: 146,692행 -> 15,794행
[병합 완료]  332,103행  x  368열


In [24]:
# 5. 파생 필드 생성 & 코드값 자연어 변환

# 통합순금액(Net): S=차변(+), H=대변(-)
df['_통합순금액'] = np.where(df['shkzg'] == 'S', df['dmbtr'], -df['dmbtr'])

# 날짜 타입 변환
df['budat'] = pd.to_datetime(df['budat'], errors='coerce')
df['bldat'] = pd.to_datetime(df['bldat'], errors='coerce')

# 회계월 (01월 ~ 12월)
df['_회계월'] = df['budat'].dt.month.apply(
    lambda m: f'{int(m):02d}월' if pd.notna(m) else np.nan
)

# 전기요일
WDAY = {0: '월요일', 1: '화요일', 2: '수요일', 3: '목요일',
        4: '금요일', 5: '토요일', 6: '일요일'}
df['_전기요일'] = df['budat'].dt.dayofweek.map(WDAY)

# 차대구분 자연어
df['_차대구분'] = df['shkzg'].map({'S': '차변(Debit)', 'H': '대변(Credit)'}).fillna(df['shkzg'])

# 전표유형(blart) 자연어
BLART_MAP = {
    'SA': '일반원장전표', 'AB': '회계전표',     'AA': '자산전표',
    'AF': '자산감가상각', 'AN': '자산취득',     'DG': '고객대변메모',
    'DR': '고객청구서',   'DZ': '고객입금',     'KA': '벤더문서',
    'KG': '벤더대변메모', 'KN': '벤더차변메모', 'KR': '벤더청구서',
    'KZ': '벤더지급',    'RE': '매입송장',     'RN': '무송장영수증',
    'WA': '출고',         'WE': '입고',         'WI': '재고문서',
    'WL': '배송출고',     'WN': '순출고',       'ZP': '지급전기',
    'ZR': '은행회계',     'ZS': '은행명세서',   'ML': '원가전기',
    'PR': '가격변경',     'GS': '증빙전표',     'RV': '청구서검증',
    'SB': '대체전표',     'EU': '유로전환',
}
df['_전표성격'] = df['blart'].map(BLART_MAP).fillna(df['blart'])

# 계정속성: ska1.xbilk == 'X' -> 재무상태표(BS), 그 외 -> 손익계산서(PL)
def map_acct_type(row):
    if row.get('xbilk') == 'X':
        return '재무상태표(BS)'
    if pd.notna(row.get('ktoks')) and str(row.get('ktoks')).upper() == 'BS':
        return '재무상태표(BS)'
    return '손익계산서(PL)'

df['_계정속성'] = df.apply(map_acct_type, axis=1)

# 역분개여부
df['_역분개여부'] = df['xreversal'].apply(
    lambda x: '역분개' if pd.notna(x) and str(x).strip() not in ['', 'nan', '0'] else '해당없음'
)

print('[파생 필드 생성 완료]')
print(f"  회계월 분포:\n{df['_회계월'].value_counts().sort_index().to_string()}")
print(f"\n  전표성격 분포:\n{df['_전표성격'].value_counts().head(10).to_string()}")


[파생 필드 생성 완료]
  회계월 분포:
_회계월
01월    38155
02월    50479
03월    82303
04월    15079
05월    11164
06월    10793
07월    34173
08월    46133
09월    11188
10월    12698
11월    10242
12월     9696

  전표성격 분포:
_전표성격
입고        126411
매입송장      123470
청구서검증      51538
배송출고       25104
출고          2602
회계전표        2277
고객입금         349
벤더청구서        124
일반원장전표        73
고객청구서         61


In [25]:
# 6. 최종 출력 스키마 구성

FINAL_SCHEMA = [
    # 내부 컬럼명       한글 컬럼명
    ('belnr',          '전표번호'),
    ('bukrs',          '회사코드'),
    ('gjahr',          '회계연도'),
    ('buzei',          '라인번호'),
    # 시간 정보
    ('budat',          '전기일'),
    ('bldat',          '증빙일'),
    ('_회계월',         '회계월'),
    ('_전기요일',       '전기요일'),
    # 계정 정보
    ('saknr',          '계정코드'),
    ('_계정명',         '계정명'),
    ('_계정속성',       '계정속성(BS/PL)'),
    # 금액 정보
    ('_통합순금액',     '통합순금액(Net)'),
    ('dmbtr',          '현지통화금액'),
    ('_차대구분',       '차대구분'),
    # 보조부 상세
    ('_통합거래처명',   '통합거래처명'),
    ('_원가센터명',     '원가센터명(부서명)'),
    ('sgtxt',          '적요'),
    ('_전표성격',       '전표성격'),
    # 상태 정보
    ('_역분개여부',     '역분개여부'),
    ('stblg',          '역분개참조번호'),
]

src_cols   = [s for s, _ in FINAL_SCHEMA]
rename_map = {s: t for s, t in FINAL_SCHEMA}

master_gl = df[src_cols].rename(columns=rename_map).copy()

print(f'[Master GL 완성]  {len(master_gl):,}행  x  {master_gl.shape[1]}열')
print()
print(master_gl.dtypes.to_string())


[Master GL 완성]  332,103행  x  20열

전표번호                   object
회사코드                   object
회계연도                   object
라인번호                   object
전기일            datetime64[ns]
증빙일            datetime64[ns]
회계월                    object
전기요일                   object
계정코드                   object
계정명                    object
계정속성(BS/PL)            object
통합순금액(Net)            float64
현지통화금액                float64
차대구분                   object
통합거래처명                 object
원가센터명(부서명)             object
적요                     object
전표성격                   object
역분개여부                  object
역분개참조번호                object


In [26]:
# 7. 데이터 정합성 검증
SEP = '=' * 55
print(SEP)
print('  데이터 정합성 검증 (Validation Protocol)')
print(SEP)

# 1. 차대 균형: 통합순금액(Net) 합계 = 0
net_sum  = master_gl['통합순금액(Net)'].sum()
balanced = abs(net_sum) < 0.01
print(f'\n[1] 차대균형 검증')
print(f'    통합순금액(Net) 합계: {net_sum:>15,.2f}')
print(f'    결과: {"OK - 균형" if balanced else "!! 불균형 - 데이터 확인 필요"}')

# 2. 행 수 유지: Master GL = 필터링된 BSEG
row_ok = (len(master_gl) == bseg_cnt)
print(f'\n[2] 행 수 유지 검증')
print(f'    필터링된 BSEG: {bseg_cnt:,}행')
print(f'    Master GL   : {len(master_gl):,}행')
print(f'    결과: {"OK - 일치" if row_ok else "!! 불일치 - 조인 중복 확인 필요"}')

# 3. 명칭 매핑율
acct_null  = master_gl['계정명'].isna().mean() * 100
prtnr_null = master_gl['통합거래처명'].isna().mean() * 100
dept_null  = master_gl['원가센터명(부서명)'].isna().mean() * 100
print(f'\n[3] 명칭 매핑율 검증')
print(f'    계정명       결측율: {acct_null:5.1f}%')
print(f'    통합거래처명 결측율: {prtnr_null:5.1f}%  (거래처 없는 전표 포함 가능)')
print(f'    원가센터명   결측율: {dept_null:5.1f}%  (원가센터 미지정 전표 포함 가능)')
print(f'\n{SEP}')


  데이터 정합성 검증 (Validation Protocol)

[1] 차대균형 검증
    통합순금액(Net) 합계:      185,402.81
    결과: !! 불균형 - 데이터 확인 필요

[2] 행 수 유지 검증
    필터링된 BSEG: 332,103행
    Master GL   : 332,103행
    결과: OK - 일치

[3] 명칭 매핑율 검증
    계정명       결측율:   0.0%
    통합거래처명 결측율:  32.0%  (거래처 없는 전표 포함 가능)
    원가센터명   결측율: 100.0%  (원가센터 미지정 전표 포함 가능)



In [27]:
# 8. 저장 & 미리보기
output_path = BASE_DIR / 'master_gl.csv'

# utf-8-sig: BOM 포함 -> 엑셀에서 한글 깨짐 없이 열림
master_gl.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'[저장 완료]  {output_path}')
print(f'파일 크기  : {output_path.stat().st_size / 1024 / 1024:.2f} MB')
print(f'행 수      : {len(master_gl):,}행  /  열 수: {master_gl.shape[1]}개')

master_gl.head(10)


[저장 완료]  G:\내 드라이브\03_DArt-B\12_정규스터디\project\master_gl.csv
파일 크기  : 62.08 MB
행 수      : 332,103행  /  열 수: 20개


,전표번호,회사코드,회계연도,라인번호,전기일,증빙일,회계월,전기요일,계정코드,계정명,계정속성(BS/PL),통합순금액(Net),현지통화금액,차대구분,통합거래처명,원가센터명(부서명),적요,전표성격,역분개여부,역분개참조번호
0,5105600151,C003,2022,002,2022-04-24,2022-04-24,04월,일요일,0000500010,Transporting,손익계산서(PL),300.0,300.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
1,5105600143,C003,2022,002,2022-04-24,2022-04-24,04월,일요일,0000500010,Transporting,손익계산서(PL),600.0,600.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
2,5105600152,C003,2022,002,2022-04-24,2022-04-24,04월,일요일,0000500010,Transporting,손익계산서(PL),550.0,550.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
3,5105600153,C003,2022,002,2022-04-25,2022-04-25,04월,월요일,0000500010,Transporting,손익계산서(PL),360.0,360.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
4,5105600154,C003,2022,002,2022-04-25,2022-04-25,04월,월요일,0000500010,Transporting,손익계산서(PL),480.0,480.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
5,5105600155,C003,2022,002,2022-04-25,2022-04-25,04월,월요일,0000500010,Transporting,손익계산서(PL),600.0,600.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
6,5105600156,C003,2022,002,2022-04-25,2022-04-25,04월,월요일,0000500010,Transporting,손익계산서(PL),420.0,420.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
7,5105600157,C003,2022,002,2022-04-25,2022-04-25,04월,월요일,0000500010,Transporting,손익계산서(PL),315.0,315.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
8,5105600158,C003,2022,002,2022-04-25,2022-04-25,04월,월요일,0000500010,Transporting,손익계산서(PL),400.0,400.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
9,5105600159,C004,2022,002,2022-04-25,2022-04-25,04월,월요일,0000500010,Transporting,손익계산서(PL),480.0,480.0,차변(Debit),NaN,NaN,NaN,매입송장,해당없음,NaN
